# Tarea 3 · Plantilla y dataset provisorio

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/3_plantilla.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Matías.

Esta plantilla carga Telco Customer Churn en local o en Colab y genera `data/processed/telco_provisorio.csv` para que las demás tareas puedan comenzar. Solo convierte `TotalCharges` a numérico: conserva los nulos y todas las filas y columnas originales. La limpieza definitiva corresponde a las tareas 6 a 9.

## Cómo ejecutarlo

- **Local:** instalar `requirements.txt` y abrir este notebook con Jupyter desde la raíz del repositorio o cualquier subcarpeta del mismo. Ejecutar todas las celdas en orden.
- **Colab:** abrir el badge y elegir **Entorno de ejecución → Ejecutar todas**. El CSV original se lee desde la URL raw de GitHub, sin montar Drive ni clonar el repositorio. El archivo generado queda en `data/processed/` dentro del directorio de trabajo de Colab y se puede descargar desde el panel Archivos.
- El badge apunta a `main` y estará disponible cuando este notebook se integre allí. Para probar una rama publicada antes del merge, reemplazar `main` por el nombre de esa rama en el enlace.


## 1. Carga del dataset original

Detectamos Colab para usar la fuente remota. En local buscamos la raíz a partir del directorio actual, lo que permite ejecutar tanto desde la raíz como desde `notebooks/trabajo/`. Conservamos una copia `df_raw` para comprobar que la conversión no cambia otras variables. Si falta el CSV local, se informa el problema en lugar de descargar otra fuente silenciosamente.


In [ ]:
from pathlib import Path

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

GITHUB_REPO = 'julianschmidt1/tpi-analisis-de-datos'
GITHUB_REF = 'main'
RUTA_RAW = Path('data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
URL_RAW = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}/{RUTA_RAW.as_posix()}'

if EN_COLAB:
    RAIZ = Path.cwd()
    fuente = URL_RAW
else:
    actual = Path.cwd().resolve()
    RAIZ = next(
        (ruta for ruta in (actual, *actual.parents) if (ruta / RUTA_RAW).is_file()),
        None,
    )
    if RAIZ is None:
        raise FileNotFoundError(
            f'No se encontró {RUTA_RAW.as_posix()}. Abrí Jupyter dentro del repositorio '
            'y comprobá que esté disponible el dataset original de la tarea 2.'
        )
    fuente = RAIZ / RUTA_RAW

df_raw = pd.read_csv(fuente)
df = df_raw.copy()
print(f'Entorno: {"Colab" if EN_COLAB else "local"}')
print(f'Fuente: {fuente}')
print(f'Dataset original: {df.shape[0]:,} filas y {df.shape[1]} columnas.')


## 2. Conversión provisoria de `TotalCharges`

`pd.to_numeric(errors='coerce')` transforma los espacios vacíos o valores no numéricos en `NaN`. Se explicita `float64` para que la columna tenga el tipo acordado. No imputamos ni eliminamos registros: esa decisión requiere la justificación de la tarea 6.

La cantidad de nulos informa cuántos clientes no tienen un total facturado utilizable todavía; los análisis con esta variable deberán tenerlo en cuenta. Este archivo aún no es el dataset limpio final.


In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').astype('float64')

assert df.shape == df_raw.shape, 'No se deben eliminar filas ni columnas.'
assert df.drop(columns='TotalCharges').equals(df_raw.drop(columns='TotalCharges')), (
    'Solo debe modificarse TotalCharges.'
)
print(f'Tipo de TotalCharges: {df["TotalCharges"].dtype}')
print(f'Nulos conservados en TotalCharges: {df["TotalCharges"].isna().sum()}')


## 3. Exportación para las siguientes tareas

Creamos el directorio de salida si hace falta y guardamos sin el índice de Pandas, para no agregar una columna artificial. Los nulos se escriben como campos vacíos. Reejecutar el notebook regenera el mismo archivo y no modifica `data/raw/`.

Los CSV no guardan metadatos de tipos; al volver a cargar este archivo con `pd.read_csv`, Pandas infiere `TotalCharges` como `float64` por sus valores numéricos y campos vacíos.


In [ ]:
RUTA_PROVISORIO = RAIZ / 'data/processed/telco_provisorio.csv'
RUTA_PROVISORIO.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RUTA_PROVISORIO, index=False, encoding='utf-8', na_rep='')
print(f'Archivo generado: {RUTA_PROVISORIO.relative_to(RAIZ).as_posix()}')
print(f'Se conservan {len(df):,} clientes y {len(df.columns)} variables para continuar el análisis.')


## Continuación del trabajo

Para una nueva tarea, crear un notebook propio en `notebooks/trabajo/<n>_<tema>.ipynb` usando esta estructura y actualizar su badge. Acompañar cada celda con una explicación de qué hace y por qué; agregar una interpretación de negocio a cada tabla o gráfico.

El archivo provisorio desbloquea el trabajo exploratorio. Cuando esté disponible `data/processed/telco_clean.csv` (tarea 9), usar esa fuente en los análisis posteriores según los contratos de `PLAN_TAREAS.md`. La integración del notebook final corresponde a la tarea 29.
